# processing_levels_pipeline

HB1603 pipeline organized into explicit processing levels.

**Author:** <Your Name>
**Version:** 1.0

In [1]:
# Uncomment the lines below to install required packages.
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_Utils.git
# %pip install echopype>=0.6.0
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_ML.git
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_Visualization.git

In [2]:
from aa_si_utils.data_retrieval import query_ncei_data
from aa_si_utils.data_retrieval import download_ncei_data
from aa_si_utils.utils import initial_setup_and_validation
from aa_si_utils.utils import read_raw_files_to_stores
from aa_si_utils.utils import combine_raw_stores
from echopype.calibrate import compute_Sv
from aa_si_utils.utils import create_surface_mask
from aa_si_utils.utils import create_frequency_mask
from echopype.consolidate import add_depth
from echopype.consolidate import add_splitbeam_angle
from echopype.mask.api import detect_seafloor
from aa_si_utils.utils import create_seafloor_mask
from aa_si_utils.utils import combine_masks
from aa_si_utils.utils import apply_mask_to_sv
from aa_si_ml.ml import remove_noise
from aa_si_visualization.echogram import plot_sv_echogram
from aa_si_utils.utils import mask_sparse_bins
from aa_si_ml.ml import compute_per_cell_statistics
from echopype.commongrid import compute_MVBS
from aa_si_ml.ml import reshape_data_for_ml
from aa_si_ml.ml import add_auxiliary_features
from aa_si_ml.ml import normalize_data
from aa_si_visualization.echogram import plot_flattened_data_echogram
from aa_si_ml.ml import run_hdbscan
from aa_si_ml.ml import embed_clustering_results
from aa_si_ml.ml import plot_clustering_report

from aa_recipe_manager.tracker.pipeline_tracker import PipelineTracker
from aa_recipe_manager.provenance.recorder import ProvenanceRecorder

c:\Users\brett.layman\Documents\AA-SI\AA-SI_recipe_manager\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import json as _json
_recipe_dict = _json.loads('{"name": "processing_levels_pipeline", "version": "1.0", "description": "HB1603 pipeline organized into explicit processing levels.", "author": "<Your Name>", "inputs": {"raw_input_folder": {"type": "path", "description": null, "default": "./raw_file_inputs", "required": false, "fingerprint_contents": false}, "cal_input_folder": {"type": "path", "description": null, "default": "./calibration_files/HB201607_cal", "required": false, "fingerprint_contents": false}, "line_file_path": {"type": "path", "description": null, "default": "./line_files/SpermWhaleClicks_click_data_HB1603_SpermWhaleDive_Span0.2_07252016_2120_UTC.csv", "required": false, "fingerprint_contents": false}, "range_bin": {"type": "str", "description": null, "default": "20m", "required": false, "fingerprint_contents": false}, "ping_time_bin": {"type": "str", "description": null, "default": "20s", "required": false, "fingerprint_contents": false}, "calibration_outputs": {"type": "str", "description": "Subdirectory name under the pipeline outputs folder for calibration artifacts.", "default": "calibration", "required": false, "fingerprint_contents": false}, "raw_file_names": {"type": "list", "description": "Optional subset of filenames within raw_input_folder to use. Empty list means \\"use every .raw file in the folder\\". Note that generate_standardized_cal_mapping always scans the full folder; this input only constrains the set of files that get opened and processed.\\n", "default": [], "required": false, "fingerprint_contents": false}}, "steps": [{"id": "query_ncei", "op": "query_ncei_data", "description": null, "inputs": {}, "params": {"file_time_start": "2021-10-12T14:20", "file_time_end": "2021-10-12T14:21", "filters": {"CRUISE_NAME": "RL2107"}}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "download_raw", "op": "download_ncei_data", "description": null, "inputs": {"results": "${query_ncei.ncei_results}", "query_label": "${query_ncei.query_label}"}, "params": {"output_dir": "${inputs.raw_input_folder}", "companion_extensions": [".bot", ".idx"]}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "initial_setup", "op": "initial_setup", "description": null, "inputs": {}, "params": {"raw_input_folder": "${download_raw.download_dir}", "calibration_outputs": "${inputs.calibration_outputs}", "raw_file_names": "${inputs.raw_file_names}", "clear_previous_json_logs": true}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "read_raw", "op": "read_raw_files", "description": null, "inputs": {"raw_file_paths": "${initial_setup.raw_file_paths}"}, "params": {"sonar_model": "EK80", "include_bot": false, "intermediate_format": "netcdf"}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "combine_raw", "op": "combine_raw_files", "description": null, "inputs": {"raw_stores": "${read_raw.raw_stores}"}, "params": {}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": "always"}, {"id": "compute_sv", "op": "compute_sv", "description": null, "inputs": {"echodata": "${combine_raw.echodata}", "waveform_mode": "CW", "encode_mode": "complex"}, "params": {}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": "always"}, {"id": "ep_add_depth", "op": "ep_add_depth", "description": null, "inputs": {"ds_Sv": "${compute_sv.ds_Sv}", "echodata": "${combine_raw.echodata}"}, "params": {"use_platform_vertical_offsets": true}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "ep_add_splitbeam_angle", "op": "ep_add_splitbeam_angle", "description": null, "inputs": {"ds_Sv": "${ep_add_depth.ds_Sv}", "echodata": "${combine_raw.echodata}"}, "params": {"waveform_mode": "CW", "encode_mode": "complex", "to_disk": false}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "detect_seafloor", "op": "ep_detect_seafloor", "description": null, "inputs": {"ds_Sv": "${ep_add_splitbeam_angle.ds_Sv}"}, "params": {"method": "basic", "method_params": {"var_name": "Sv", "channel": "GPT  38 kHz ..."}}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "create_seafloor_mask", "op": "create_seafloor_mask", "description": null, "inputs": {"ds_Sv": "${ep_add_splitbeam_angle.ds_Sv}", "seafloor_depth": "${detect_seafloor.seafloor_depth}"}, "params": {"seafloor_buffer_m": 100}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "create_surface_mask", "op": "create_surface_mask", "description": null, "inputs": {"ds_Sv": "${compute_sv.ds_Sv}"}, "params": {"surface_depth_m": 10.0}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "create_frequency_mask", "op": "create_frequency_mask", "description": null, "inputs": {"ds_Sv": "${compute_sv.ds_Sv}"}, "params": {"frequencies_to_mask": [70, 120, 200]}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "combine_masks", "op": "combine_masks", "description": null, "inputs": {"masks": ["${create_seafloor_mask.mask}", "${create_surface_mask.mask}", "${create_frequency_mask.mask}"]}, "params": {"mode": "and"}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "apply_mask", "op": "apply_sv_mask", "description": null, "inputs": {"ds_Sv": "${compute_sv.ds_Sv}", "mask": "${combine_masks.mask}"}, "params": {}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "remove_noise", "op": "remove_background_noise", "description": null, "inputs": {"ds_Sv": "${apply_mask.ds_Sv}"}, "params": {"range_sample_num": 10, "ping_num": 5}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": "always"}, {"id": "plot_sv_clean", "op": "plot_sv_echogram", "description": null, "inputs": {"ds_Sv": "${remove_noise.ds_Sv}"}, "params": {"min_depth": 0, "max_depth": 1800, "ping_min": 0, "ping_max": 515, "x_axis_units": "seconds", "y_axis_units": "meters"}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "mask_sparse", "op": "mask_sparse_bins", "description": null, "inputs": {"ds_Sv": "${remove_noise.ds_Sv}"}, "params": {"range_bin": "${inputs.range_bin}", "ping_time_bin": "${inputs.ping_time_bin}", "nan_threshold": 0.6}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "compute_mvbs", "op": "compute_mvbs", "description": null, "inputs": {"ds_Sv": "${mask_sparse.ds_Sv}"}, "params": {"range_bin": "${inputs.range_bin}", "ping_time_bin": "${inputs.ping_time_bin}"}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": "always"}, {"id": "compute_cell_stats", "op": "compute_per_cell_statistics", "description": null, "inputs": {"ds_Sv": "${remove_noise.ds_Sv}"}, "params": {"range_bin": "${inputs.range_bin}", "ping_time_bin": "${inputs.ping_time_bin}", "statistics": ["cv"]}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "plot_mvbs", "op": "plot_sv_echogram", "description": null, "inputs": {"ds_Sv": "${compute_mvbs.ds_MVBS}", "ds_Sv_source": "${mask_sparse.ds_Sv}"}, "params": {"min_depth": 0, "max_depth": 1800, "ping_min": 0, "ping_max": 515, "x_axis_units": "seconds", "y_axis_units": "meters", "overlay_lines": [{"var": "sw_dive_profile_fit", "style": {"color": "red", "linewidth": 3.5}}]}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "reshape_ml", "op": "reshape_for_ml", "description": null, "inputs": {"ds_MVBS": "${compute_mvbs.ds_MVBS}"}, "params": {"data_var": "Sv", "dataset_name": "ml_dataset", "feature_strategy": "mean_centered", "baseline_channel": 0}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "add_aux_features", "op": "add_auxiliary_features", "description": null, "inputs": {"ds_ml": "${reshape_ml.ds_ml_ready}", "echodata": "${combine_raw.echodata}", "ds_sv": "${compute_cell_stats.ds_Sv}"}, "params": {"dataset_name": "ml_dataset", "features": ["cell_cv"]}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "normalize_ml", "op": "normalize_ml_data", "description": null, "inputs": {"ds_ml": "${add_aux_features.ds_ml_ready}"}, "params": {"method": "flatten", "dataset_name": "ml_dataset", "normalization_name": "normalized_flatten_mean_centered", "shift_positive": false}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": "always"}, {"id": "plot_normalized_ml", "op": "plot_ml_echogram", "description": null, "inputs": {"ds_normalized": "${normalize_ml.ds_normalized}", "ds_Sv": "${remove_noise.ds_Sv}"}, "params": {"dataset_name": "ml_dataset", "normalization_name": "normalized_flatten_mean_centered", "min_depth": 0, "max_depth": 1800, "ping_min": 0, "ping_max": 515, "x_axis_units": "seconds", "y_axis_units": "meters"}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "run_hdbscan", "op": "run_hdbscan", "description": null, "inputs": {"ds_normalized": "${normalize_ml.ds_normalized}"}, "params": {"dataset_name": "ml_dataset", "normalization_name": "normalized_flatten_mean_centered", "ml_result_name": "hdbscan_results", "min_cluster_size_fraction": 0.02, "min_samples": 10, "sample_size": 1000000, "cluster_selection_method": "leaf", "use_hdbscan": true}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "embed_results", "op": "embed_clustering_results", "description": null, "inputs": {"ds_normalized": "${normalize_ml.ds_normalized}", "clustering_results": "${run_hdbscan.clustering_results}"}, "params": {"dataset_name": "ml_dataset", "ml_result_name": "hdbscan_results"}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}, {"id": "plot_clustering_report", "op": "plot_clustering_report", "description": null, "inputs": {"ds_normalized": "${embed_results.ds_normalized}", "clustering_results": "${run_hdbscan.clustering_results}", "clustering_model": "${run_hdbscan.clustering_model}", "ds_Sv": "${remove_noise.ds_Sv}"}, "params": {"dataset_name": "ml_dataset", "ml_result_name": "hdbscan_results", "plot_window": [0, 1800, 0, 515], "overlay_line_var": "sw_dive_profile", "cluster_colors": ["#9D00FF", "#35E200", "#FF0000", "#2F00FF", "#FF8800", "#FFFB1C", "#FF44E9", "#00BE8BCA", "#00FFE5", "#3D8D00FF", "#9C67FFFF"], "cluster_stats_sv_data_var": "Sv"}, "depends_on": null, "implementation_override": null, "custom_spec": null, "map_over": null, "collect": null, "sweep": null, "execution": null, "checkpoint": null}], "outputs": null, "execution": null, "include_blocks": [{"source": "processing_lvl_0.yaml", "step_ids": ["query_ncei", "download_raw"]}, {"source": "processing_lvl_1.yaml", "step_ids": ["initial_setup", "read_raw", "combine_raw"]}, {"source": "processing_lvl_2.yaml", "step_ids": ["compute_sv", "ep_add_depth", "ep_add_splitbeam_angle", "detect_seafloor", "create_seafloor_mask", "create_surface_mask", "create_frequency_mask", "combine_masks", "apply_mask", "remove_noise"]}, {"source": "processing_lvl_3.yaml", "step_ids": ["mask_sparse", "compute_mvbs", "compute_cell_stats"]}, {"source": "processing_lvl_4.yaml", "step_ids": ["reshape_ml", "add_aux_features", "normalize_ml", "plot_normalized_ml", "run_hdbscan", "embed_results", "plot_clustering_report"]}], "schema_version": "1"}')
tracker = PipelineTracker(_recipe_dict)

In [4]:
raw_input_folder = './raw_file_inputs'  # path
cal_input_folder = './calibration_files/HB201607_cal'  # path
line_file_path = './line_files/SpermWhaleClicks_click_data_HB1603_SpermWhaleDive_Span0.2_07252016_2120_UTC.csv'  # path
range_bin = '20m'  # str
ping_time_bin = '20s'  # str
calibration_outputs = 'calibration'  # str
raw_file_names = []  # list

## Included: processing_lvl_0.yaml

Steps: query_ncei, download_raw

### Step: `query_ncei`
**Op:** `query_ncei_data`

Query an NCEI acoustic data catalog for file listings within a time window. Returns a list of result records (file metadata), not the files themselves.

**Parameters:**
- `file_time_start` str - ISO 8601 start of window, e.g. '2016-07-25T20:58'
- `file_time_end` str - ISO 8601 end of window
- `filters` dict - Key-value filters applied to NCEI metadata (e.g. CRUISE_NAME)

In [5]:
# --- Parameters ---
_query_ncei__file_time_start = '2021-10-12T14:20'
_query_ncei__file_time_end = '2021-10-12T14:21'

with tracker.step('query_ncei', op='query_ncei_data', params={
    'file_time_start': _query_ncei__file_time_start,
    'file_time_end': _query_ncei__file_time_end,
}):
    _result = query_ncei_data(
    file_time_start=_query_ncei__file_time_start,
    file_time_end=_query_ncei__file_time_end,
    filters={'CRUISE_NAME': 'RL2107'},
)
    ncei_results = _result['records']
    query_label = _result['query_label']

Querying NCEI WCSD archive...
  Filename-time filter: 324 -> 1 results (2021-10-12 14:20:00 to 2021-10-12 14:21:00)
Query returned 1 result(s). Label: RL2107_2021-10-12_1420-1421
  2107RL_CW-D20211012-T142056.tar


### Step: `download_raw`
**Op:** `download_ncei_data`

Download raw acoustic files identified by an NCEI query result, along with any specified companion file extensions (.bot, .idx, etc.).

**Parameters:**
- `output_dir` path - Local *base* directory. Each query writes into output_dir/<query_id or query_label>/.

- `companion_extensions` list - Additional file extensions to download alongside each .raw file
- `query_id` str - Optional explicit subfolder name. Overrides the query_label input. Useful when you want a stable, human-chosen folder name.


In [6]:
# --- Parameters ---
_download_raw__output_dir = './raw_file_inputs'
_download_raw__companion_extensions = ['.bot', '.idx']

with tracker.step('download_raw', op='download_ncei_data', params={
    'output_dir': _download_raw__output_dir,
    'companion_extensions': _download_raw__companion_extensions,
}):
    _result = download_ncei_data(
    results=ncei_results,
    query_label=query_label,
    output_dir=_download_raw__output_dir,
    companion_extensions=_download_raw__companion_extensions,
)
    downloaded_paths = _result['downloaded_paths']
    download_dir = _result['download_dir']

  [1/1] Already exists, skipping: 2107RL_CW-D20211012-T142056.raw
Download complete. 2 file(s) ready in raw_file_inputs\RL2107_2021-10-12_1420-1421


*End of included section: processing_lvl_0.yaml*

## Included: processing_lvl_1.yaml

Steps: initial_setup, read_raw, combine_raw

### Step: `initial_setup`
**Op:** `initial_setup`

Validate the raw file set in the input directory, create calibration output directories if they do not exist, and return a list of raw file paths and the resolved calibration output directory ready for processing.

**Parameters:**
- `raw_input_folder` path
- `calibration_outputs` str - Subdirectory name under the pipeline's user-facing outputs folder where calibration artifacts are written. Resolved automatically relative to the outputs directory when run via the executor; treated as a CWD-relative path when called standalone (e.g. in a notebook). Calibration JSON logs are always written to a logs/ subdirectory within this folder.
- `raw_file_names` list - Explicit file name list; empty means include all files in folder
- `clear_previous_json_logs` bool

In [7]:
# --- Parameters ---
_initial_setup__raw_input_folder = download_dir
_initial_setup__calibration_outputs = 'calibration'
_initial_setup__raw_file_names = []
_initial_setup__clear_previous_json_logs = True

with tracker.step('initial_setup', op='initial_setup', params={
    'raw_input_folder': _initial_setup__raw_input_folder,
    'calibration_outputs': _initial_setup__calibration_outputs,
    'raw_file_names': _initial_setup__raw_file_names,
    'clear_previous_json_logs': _initial_setup__clear_previous_json_logs,
}):
    _result = initial_setup_and_validation(
    raw_input_folder=_initial_setup__raw_input_folder,
    calibration_outputs_string=_initial_setup__calibration_outputs,
    raw_file_names=_initial_setup__raw_file_names,
    clear_previous_json_logs=_initial_setup__clear_previous_json_logs,
)
    raw_file_paths = _result['raw_file_paths']
    calibration_output_dir = _result['calibration_output_dir']

Found 1 raw file(s) to process:
  - 2107RL_CW-D20211012-T142056.raw
Created missing output folder: calibration\logs


### Step: `read_raw`
**Op:** `read_raw_files`

Open raw echosounder files with echopype and write each file as an intermediate per-file store. Returns a list of store-path strings for file-backed formats, or a list of in-memory EchoData objects when intermediate_format is "none". This is the future map_over target for parallel per-file processing. NOTE: use intermediate_format "netcdf" (default) or "none". The "zarr" option is not recommended — see the intermediate_format parameter description below.

**Parameters:**
- `sonar_model` str - Sonar model string recognized by echopype (e.g. 'EK60', 'EK80')
- `include_bot` bool - Include bottom-detection (.bot) data if available
- `intermediate_format` str - Intermediate store format written to exe_temp/data between the read_raw and combine_raw steps. Supported values: "netcdf" (default), "zarr", "none".
"netcdf" (default, recommended): each raw file is written as a temporary .nc file, reopened, and combined. After combining, all EchoData groups are rechunked to a single uniform chunk so the downstream zarr v2 checkpoint write (combine_raw with checkpoint: always) always succeeds.
"none": all EchoData objects are kept in memory with no temp files. Use when the full dataset fits comfortably in RAM.
"zarr" (NOT recommended): zarr v3 intermediate stores use serializer-backed dtypes that are incompatible with the zarr v2 checkpoint writer used for EchoData checkpoints. Using this option will reproduce the error: "Zarr format 2 arrays do not support serializer". Kept as an option for experimentation only.

- `use_swap` str - Passed to echopype open_raw as use_swap. "auto" (default) triggers per-file swap when the expanded data would exceed ~40% of total RAM.


In [8]:
# --- Parameters ---
_read_raw__sonar_model = 'EK80'
_read_raw__include_bot = False
_read_raw__intermediate_format = 'netcdf'

with tracker.step('read_raw', op='read_raw_files', params={
    'sonar_model': _read_raw__sonar_model,
    'include_bot': _read_raw__include_bot,
    'intermediate_format': _read_raw__intermediate_format,
}):
    raw_stores = read_raw_files_to_stores(
    raw_file_paths=raw_file_paths,
    sonar_model=_read_raw__sonar_model,
    include_bot=_read_raw__include_bot,
    intermediate_format=_read_raw__intermediate_format,
)

Read 1 raw file(s) to 'netcdf' format


### Step: `combine_raw`
**Op:** `combine_raw_files`

Lazily open per-file intermediate stores (zarr or netcdf paths) produced by read_raw_files, or combine in-memory EchoData objects (none mode), into a single combined EchoData object. Mark this step checkpoint: always in your recipe so the checkpoint system writes the combined output once and reuses it on subsequent runs.

In [9]:
with tracker.step('combine_raw', op='combine_raw_files', params={}):
    echodata = combine_raw_stores(raw_stores=raw_stores)

EchoData ready for processing


*End of included section: processing_lvl_1.yaml*

## Included: processing_lvl_2.yaml

Steps: compute_sv, ep_add_depth, ep_add_splitbeam_angle, detect_seafloor, create_seafloor_mask, create_surface_mask, create_frequency_mask, combine_masks, apply_mask, remove_noise

### Step: `compute_sv`
**Op:** `compute_sv`

Compute volume backscattering strength (Sv) from raw EchoData. When cal_params and env_params are supplied they are applied to the computation. When omitted, echopype uses default environmental and calibration values embedded in the raw file.

In [10]:
with tracker.step('compute_sv', op='compute_sv', params={}):
    compute_sv_ds_Sv = compute_Sv(
    echodata=echodata,
    waveform_mode='CW',
    encode_mode='complex',
)

### Step: `ep_add_depth`
**Op:** `ep_add_depth`

Add a surface-referenced ``depth`` data variable to an Sv Dataset using echopype's ``consolidate.add_depth``. The result preserves all existing variables (including ``echo_range``) and adds ``depth`` on ``(channel, ping_time, range_sample)`` in metres below surface.
Whatever depth-computation technique is selected (constant ``depth_offset``, Platform vertical offsets, Platform/Beam angles, user-supplied ``tilt``) is fully captured in the values of the resulting ``depth`` array, so downstream consumers can treat the dataset as the single source of truth for vertical reference frame without knowing which technique was used.

**Parameters:**
- `depth_offset` float (m) - Constant vertical offset for the transducer position. When set, takes precedence over Platform vertical offsets.

- `tilt` float (degree) - Constant transducer tilt; 0 corresponds to a vertically-pointing transducer. When set, takes precedence over Platform/Beam angles.

- `downward` bool - True when transducers point downward (typical hull-mounted).
- `use_platform_vertical_offsets` bool - Use Echodata Platform group vertical offsets to compute transducer depth. EK60/EK80 only. Ignored when ``depth_offset`` is set.

- `use_platform_angles` bool - Use Echodata Platform group angles to compute echo_range scaling. EK60/EK80 only. Cannot be combined with ``use_beam_angles``.

- `use_beam_angles` bool - Use Echodata Beam group angles to compute echo_range scaling. EK60/EK80 only. Cannot be combined with ``use_platform_angles``.


In [11]:
# --- Parameters ---
_ep_add_depth__use_platform_vertical_offsets = True

with tracker.step('ep_add_depth', op='ep_add_depth', params={
    'use_platform_vertical_offsets': _ep_add_depth__use_platform_vertical_offsets,
}):
    ep_add_depth_ds_Sv = add_depth(
    ds=compute_sv_ds_Sv,
    echodata=echodata,
    use_platform_vertical_offsets=_ep_add_depth__use_platform_vertical_offsets,
)

### Step: `ep_add_splitbeam_angle`
**Op:** `ep_add_splitbeam_angle`

Add split-beam alongship and athwartship angle variables to an Sv Dataset using echopype's ``consolidate.add_splitbeam_angle``. The result preserves the input Sv data and adds ``angle_alongship`` and ``angle_athwartship`` on ``(channel, ping_time, range_sample)`` when the source EchoData contains the required split-beam angle data.
This step is required before echopype's Blackwell seafloor detector because that detector thresholds the per-sample split-beam angle fields. Calibration parameters such as ``angle_offset_alongship`` are used by echopype to compute these variables, but they are not themselves the angle fields consumed by the detector.

**Parameters:**
- `waveform_mode` str - Transmit waveform mode. Use ``CW`` for narrowband EK60/EK80 power/angle data, or ``BB`` for broadband complex samples.

- `encode_mode` str - Return echo encoding mode. Use ``power`` for EK60/EK80 power/angle data, or ``complex`` for complex samples.

- `pulse_compression` bool - Whether to pulse-compress broadband complex samples. Only valid for ``waveform_mode: BB`` and ``encode_mode: complex``.

- `to_disk` bool - Write split-beam angles back to disk when ``ds_Sv`` is a path. Keep false for in-memory recipe execution.

- `drop_last_hanning_zero` bool - Echopype compatibility option for broadband pulse compression. Usually leave false.


In [12]:
# --- Parameters ---
_ep_add_splitbeam_angle__waveform_mode = 'CW'
_ep_add_splitbeam_angle__encode_mode = 'complex'
_ep_add_splitbeam_angle__to_disk = False

with tracker.step('ep_add_splitbeam_angle', op='ep_add_splitbeam_angle', params={
    'waveform_mode': _ep_add_splitbeam_angle__waveform_mode,
    'encode_mode': _ep_add_splitbeam_angle__encode_mode,
    'to_disk': _ep_add_splitbeam_angle__to_disk,
}):
    ep_add_splitbeam_angle_ds_Sv = add_splitbeam_angle(
    source_Sv=ep_add_depth_ds_Sv,
    echodata=echodata,
    waveform_mode=_ep_add_splitbeam_angle__waveform_mode,
    encode_mode=_ep_add_splitbeam_angle__encode_mode,
    to_disk=_ep_add_splitbeam_angle__to_disk,
)

KeyboardInterrupt: 

In [14]:
ep_add_depth_ds_Sv.channel

<xarray.DataArray 'channel' (channel: 5)> Size: 500B
array(['WBT 987753-15 ES120-7C_ES', 'WBT 987763-15 ES38-7_ES',
       'WBT 987766-15 ES70-7C_ES', 'WBT 987769-15 ES333-7C_ES',
       'WBT 987771-15 ES200-7C_ES'], dtype='<U25')
Coordinates:
  * channel  (channel) <U25 500B 'WBT 987753-15 ES120-7C_ES' ... 'WBT 987771-...
Attributes:
    long_name:  Vendor channel ID

### Step: `detect_seafloor`
**Op:** `ep_detect_seafloor`

Detect seafloor depth using echopype's bottom detection dispatcher and return a 1-D (ping_time,) DataArray. Dispatches to one of echopype's built-in implementations ("basic" Sv-threshold or "blackwell" Sv + split-beam angle) selected by the ``method`` parameter, with the method-specific keyword arguments passed via ``method_params``.
Drop-in swap with the ``detect_seafloor`` op: the output port name (``seafloor_depth``) and shape (1-D, ping_time) match, so downstream steps such as ``create_seafloor_mask`` work unchanged. The recipe step's ``inputs`` must drop the ``echodata`` line (echopype's detector does not use a separate EchoData source) and the ``params`` block must use ``method`` + ``method_params`` instead of ``channel`` / ``min_valid_depth_m``.

**Parameters:**
- `method` str - Echopype bottom-detection method. Supported values:
  - "basic": Sv threshold-only detector.
  - "blackwell": Sv + split-beam angle detector.

- `method_params` dict - Method-specific keyword arguments forwarded to the chosen echopype detector. Unspecified keys fall back to that method's defaults.
basic:
  var_name (str): Name of Sv variable (dB), e.g. "Sv".
  channel (str): Channel identifier to select.
  threshold (float | (float, float), default -50.0): Sv threshold in
    dB; a single value is treated as the lower bound (upper = lower
    + 10 dB), or pass a 2-tuple (tmin, tmax) for both bounds.
  offset_m (float, default 0.5): Metres subtracted from the detected
    crossing.
  bin_skip_from_surface (int, default 200): Number of shallow range
    bins to ignore before searching.

blackwell:
  var_name (str): Name of the Sv variable to use.
  channel (str): Channel identifier to select.
  threshold (float | (float, float, float), default -75): Either a
    single Sv dB threshold (angle thresholds use defaults), or a
    3-tuple (tSv_dB, ttheta, tphi) after angle smoothing.
  offset (float, default 0.3): Metres subtracted from the detected
    bottom.
  r0 (float, default 0): Shallow bound (m) of the detection range.
  r1 (float, default 500): Deep bound (m) of the detection range.
  wtheta (int, default 28): Square smoothing window (pixels) for
    along-ship angle.
  wphi (int, default 52): Square smoothing window (pixels) for
    athwart-ship angle.


In [ ]:
ep_add_splitbeam_angle_ds_Sv

In [ ]:
# --- Parameters ---
_detect_seafloor__method = 'basic'

with tracker.step('detect_seafloor', op='ep_detect_seafloor', params={
    'method': _detect_seafloor__method,
}):
    seafloor_depth = detect_seafloor(
    ds=ep_add_splitbeam_angle_ds_Sv,
    method=_detect_seafloor__method,
    params={'var_name': 'Sv', 'channel': 'GPT  38 kHz ...'},
)

### Step: `create_seafloor_mask`
**Op:** `create_seafloor_mask`

Build a seafloor mask from baseline Sv and detected seafloor depth. The seafloor depth is passed as an explicit input, enabling swappable detection methods without modifying masking logic.

**Parameters:**
- `seafloor_buffer_m` float (m)
- `range_var` str - Name of the meter-valued variable on ds_Sv to compare against the seafloor line. Must use the same vertical reference as the seafloor line. When null (the default), auto-selects "depth" if present on ds_Sv (i.e., add_depth was run upstream), otherwise falls back to "echo_range".


In [ ]:
# --- Parameters ---
_create_seafloor_mask__seafloor_buffer_m = 100

with tracker.step('create_seafloor_mask', op='create_seafloor_mask', params={
    'seafloor_buffer_m': _create_seafloor_mask__seafloor_buffer_m,
}):
    create_seafloor_mask_mask = create_seafloor_mask(
    ds_Sv=ep_add_splitbeam_angle_ds_Sv,
    seafloor_depth=seafloor_depth,
    seafloor_buffer_m=_create_seafloor_mask__seafloor_buffer_m,
)

### Step: `create_surface_mask`
**Op:** `create_surface_mask`

Build a surface-interference mask from baseline Sv using a selectable method.

**Parameters:**
- `surface_depth_m` float (m)

In [ ]:
# --- Parameters ---
_create_surface_mask__surface_depth_m = 10.0

with tracker.step('create_surface_mask', op='create_surface_mask', params={
    'surface_depth_m': _create_surface_mask__surface_depth_m,
}):
    create_surface_mask_mask = create_surface_mask(ds_Sv=compute_sv_ds_Sv, surface_depth_m=_create_surface_mask__surface_depth_m)

### Step: `create_frequency_mask`
**Op:** `create_frequency_mask`

Build a mask that excludes selected frequency channels. Implementation can be swapped to use different exclusion criteria.

**Parameters:**
- `frequencies_to_mask` list (kHz)

In [ ]:
# --- Parameters ---
_create_frequency_mask__frequencies_to_mask = [70, 120, 200]

with tracker.step('create_frequency_mask', op='create_frequency_mask', params={
    'frequencies_to_mask': _create_frequency_mask__frequencies_to_mask,
}):
    create_frequency_mask_mask = create_frequency_mask(ds_Sv=compute_sv_ds_Sv, frequencies_to_mask=_create_frequency_mask__frequencies_to_mask)

### Step: `combine_masks`
**Op:** `combine_masks`

Combine two or more mask datasets into one final mask. Accepts a list of mask inputs to support inserting additional custom mask-producing steps into the chain without changing core framework code.

**Parameters:**
- `mode` str - Combination mode: 'and' (all must be valid) or 'or' (any must be valid)

In [ ]:
# --- Parameters ---
_combine_masks__mode = 'and'

with tracker.step('combine_masks', op='combine_masks', params={
    'mode': _combine_masks__mode,
}):
    combine_masks_mask = combine_masks(masks=[create_seafloor_mask_mask, create_surface_mask_mask, create_frequency_mask_mask], mode=_combine_masks__mode)

### Step: `apply_mask`
**Op:** `apply_sv_mask`

Apply a boolean mask to an Sv Dataset, setting masked samples to NaN.

In [ ]:
with tracker.step('apply_mask', op='apply_sv_mask', params={}):
    apply_mask_ds_Sv = apply_mask_to_sv(ds_Sv=compute_sv_ds_Sv, mask=combine_masks_mask)

### Step: `remove_noise`
**Op:** `remove_background_noise`

Estimate and remove background noise from Sv using the De Robertis and Higginbottom method. Returns a cleaned Sv Dataset.

**Parameters:**
- `ping_num` int - Number of pings used to estimate the noise level
- `range_sample_num` int - Number of range samples used to estimate the noise level
- `background_noise_max` str - Upper limit on the estimated noise level (e.g. '-125dB'); omit to leave it unbounded
- `SNR_threshold` str - Signal-to-noise ratio below which corrected samples are masked out (e.g. '3.0dB')

In [ ]:
# --- Parameters ---
_remove_noise__range_sample_num = 10
_remove_noise__ping_num = 5

with tracker.step('remove_noise', op='remove_background_noise', params={
    'range_sample_num': _remove_noise__range_sample_num,
    'ping_num': _remove_noise__ping_num,
}):
    remove_noise_ds_Sv = remove_noise(
    ds_Sv=apply_mask_ds_Sv,
    range_sample_num=_remove_noise__range_sample_num,
    ping_num=_remove_noise__ping_num,
)

*End of included section: processing_lvl_2.yaml*

## Step: `plot_sv_clean`
**Op:** `plot_sv_echogram`

Render an Sv echogram. Supports two call patterns: (1) single dataset, plots ds_Sv directly; (2) dual dataset, plots ds_Sv using coordinate axes from ds_Sv_source (useful for MVBS overlaid on the original Sv grid). This is a sink step: it produces a side-effect (figure) with no data output.

**Parameters:**
- `min_depth` float - Minimum depth (y-axis) in meters
- `max_depth` float - Maximum depth (y-axis) in meters
- `ping_min` int - First ping index to display
- `ping_max` int - Last ping index to display
- `x_axis_units` str - Units for the x-axis ('seconds' or 'datetime')
- `y_axis_units` str - Units for the y-axis ('meters' or 'samples')
- `overlay_lines` list - Optional list of line overlays. Each entry is a dict with 'var' (variable name in ds_Sv) and 'style' (dict of matplotlib kwargs).

- `save_image` bool - Save the rendered figure to an image file; null uses runtime defaults
- `save_formats` list - Image formats to save, chosen from png, svg, jpeg, jpg, and pdf
- `save_dir` str - Optional directory override for saved image artifacts
- `show` bool - Whether to display the figure interactively or inline; null uses runtime defaults
- `dpi` int - Optional DPI override for raster image outputs

In [ ]:
# --- Parameters ---
_plot_sv_clean__min_depth = 0
_plot_sv_clean__max_depth = 1800
_plot_sv_clean__ping_min = 0
_plot_sv_clean__ping_max = 515
_plot_sv_clean__x_axis_units = 'seconds'
_plot_sv_clean__y_axis_units = 'meters'

with tracker.step('plot_sv_clean', op='plot_sv_echogram', params={
    'min_depth': _plot_sv_clean__min_depth,
    'max_depth': _plot_sv_clean__max_depth,
    'ping_min': _plot_sv_clean__ping_min,
    'ping_max': _plot_sv_clean__ping_max,
    'x_axis_units': _plot_sv_clean__x_axis_units,
    'y_axis_units': _plot_sv_clean__y_axis_units,
}):
    plot_sv_echogram(
    ds_Sv=remove_noise_ds_Sv,
    min_depth=_plot_sv_clean__min_depth,
    max_depth=_plot_sv_clean__max_depth,
    ping_min=_plot_sv_clean__ping_min,
    ping_max=_plot_sv_clean__ping_max,
    x_axis_units=_plot_sv_clean__x_axis_units,
    y_axis_units=_plot_sv_clean__y_axis_units,
)

## Included: processing_lvl_3.yaml

Steps: mask_sparse, compute_mvbs, compute_cell_stats

### Step: `mask_sparse`
**Op:** `mask_sparse_bins`

Remove spatiotemporal bins with fewer valid samples than a threshold, reducing noise in MVBS computation.

**Parameters:**
- `range_bin` str - Range bin size (e.g. '2m')
- `ping_time_bin` str - Ping-time bin size (e.g. '10s')
- `nan_threshold` float - Fraction of NaN values above which a bin is masked (0.0 to 1.0)

In [ ]:
# --- Parameters ---
_mask_sparse__range_bin = '20m'
_mask_sparse__ping_time_bin = '20s'
_mask_sparse__nan_threshold = 0.6

with tracker.step('mask_sparse', op='mask_sparse_bins', params={
    'range_bin': _mask_sparse__range_bin,
    'ping_time_bin': _mask_sparse__ping_time_bin,
    'nan_threshold': _mask_sparse__nan_threshold,
}):
    mask_sparse_ds_Sv = mask_sparse_bins(
    ds_Sv=remove_noise_ds_Sv,
    range_bin=_mask_sparse__range_bin,
    ping_time_bin=_mask_sparse__ping_time_bin,
    nan_threshold=_mask_sparse__nan_threshold,
)

### Step: `compute_mvbs`
**Op:** `compute_mvbs`

Compute Mean Volume Backscattering Strength by averaging Sv over range and time bins. Returns a gridded MVBS Dataset.

**Parameters:**
- `range_bin` str - Depth bin size for averaging (e.g. '2m')
- `ping_time_bin` str - Ping-time bin size for averaging (e.g. '10s')

In [ ]:
# --- Parameters ---
_compute_mvbs__range_bin = '20m'
_compute_mvbs__ping_time_bin = '20s'

with tracker.step('compute_mvbs', op='compute_mvbs', params={
    'range_bin': _compute_mvbs__range_bin,
    'ping_time_bin': _compute_mvbs__ping_time_bin,
}):
    ds_MVBS = compute_MVBS(
    ds_Sv=mask_sparse_ds_Sv,
    range_bin=_compute_mvbs__range_bin,
    ping_time_bin=_compute_mvbs__ping_time_bin,
)

### Step: `compute_cell_stats`
**Op:** `compute_per_cell_statistics`

Compute per-MVBS-cell statistics (e.g. coefficient of variation) from fine-resolution Sv data before masking or MVBS averaging. Results are stored as new DataArrays in the returned dataset at MVBS-bin resolution. Must be called with the same range_bin and ping_time_bin values used by mask_sparse_bins and compute_mvbs so that bin grids align exactly. Pass the returned dataset to add_auxiliary_features via the ds_sv input to absorb the statistics into the ML feature matrix.

**Parameters:**
- `range_bin` str - Range bin size — must match mask_sparse_bins and compute_mvbs (e.g. '2m')
- `ping_time_bin` str - Ping-time bin size — must match mask_sparse_bins and compute_mvbs (e.g. '10s')
- `statistics` list - List of statistics to compute. Supported: ['cv']
- `data_var` str - Name of the Sv variable in the input Dataset

In [ ]:
# --- Parameters ---
_compute_cell_stats__range_bin = '20m'
_compute_cell_stats__ping_time_bin = '20s'
_compute_cell_stats__statistics = ['cv']

with tracker.step('compute_cell_stats', op='compute_per_cell_statistics', params={
    'range_bin': _compute_cell_stats__range_bin,
    'ping_time_bin': _compute_cell_stats__ping_time_bin,
    'statistics': _compute_cell_stats__statistics,
}):
    compute_cell_stats_ds_Sv = compute_per_cell_statistics(
    ds_Sv=remove_noise_ds_Sv,
    range_bin=_compute_cell_stats__range_bin,
    ping_time_bin=_compute_cell_stats__ping_time_bin,
    statistics=_compute_cell_stats__statistics,
)

*End of included section: processing_lvl_3.yaml*

## Step: `plot_mvbs`
**Op:** `plot_sv_echogram`

Render an Sv echogram. Supports two call patterns: (1) single dataset, plots ds_Sv directly; (2) dual dataset, plots ds_Sv using coordinate axes from ds_Sv_source (useful for MVBS overlaid on the original Sv grid). This is a sink step: it produces a side-effect (figure) with no data output.

**Parameters:**
- `min_depth` float - Minimum depth (y-axis) in meters
- `max_depth` float - Maximum depth (y-axis) in meters
- `ping_min` int - First ping index to display
- `ping_max` int - Last ping index to display
- `x_axis_units` str - Units for the x-axis ('seconds' or 'datetime')
- `y_axis_units` str - Units for the y-axis ('meters' or 'samples')
- `overlay_lines` list - Optional list of line overlays. Each entry is a dict with 'var' (variable name in ds_Sv) and 'style' (dict of matplotlib kwargs).

- `save_image` bool - Save the rendered figure to an image file; null uses runtime defaults
- `save_formats` list - Image formats to save, chosen from png, svg, jpeg, jpg, and pdf
- `save_dir` str - Optional directory override for saved image artifacts
- `show` bool - Whether to display the figure interactively or inline; null uses runtime defaults
- `dpi` int - Optional DPI override for raster image outputs

In [ ]:
# --- Parameters ---
_plot_mvbs__min_depth = 0
_plot_mvbs__max_depth = 1800
_plot_mvbs__ping_min = 0
_plot_mvbs__ping_max = 515
_plot_mvbs__x_axis_units = 'seconds'
_plot_mvbs__y_axis_units = 'meters'

with tracker.step('plot_mvbs', op='plot_sv_echogram', params={
    'min_depth': _plot_mvbs__min_depth,
    'max_depth': _plot_mvbs__max_depth,
    'ping_min': _plot_mvbs__ping_min,
    'ping_max': _plot_mvbs__ping_max,
    'x_axis_units': _plot_mvbs__x_axis_units,
    'y_axis_units': _plot_mvbs__y_axis_units,
}):
    plot_sv_echogram(
    ds_Sv=ds_MVBS,
    ds_Sv_original=mask_sparse_ds_Sv,
    min_depth=_plot_mvbs__min_depth,
    max_depth=_plot_mvbs__max_depth,
    ping_min=_plot_mvbs__ping_min,
    ping_max=_plot_mvbs__ping_max,
    x_axis_units=_plot_mvbs__x_axis_units,
    y_axis_units=_plot_mvbs__y_axis_units,
    overlay_lines=[{'var': 'sw_dive_profile_fit', 'style': {'color': 'red', 'linewidth': 3.5}}],
)

## Included: processing_lvl_4.yaml

Steps: reshape_ml, add_aux_features, normalize_ml, plot_normalized_ml, run_hdbscan, embed_results, plot_clustering_report

### Step: `reshape_ml`
**Op:** `reshape_for_ml`

Reshape an MVBS Dataset into a flat, ML-ready xarray Dataset where each observation corresponds to one spatial or temporal bin.

**Parameters:**
- `data_var` str - Name of the data variable to use as features
- `dataset_name` str - Label assigned to this ML dataset for tracking
- `feature_strategy` str - Feature construction strategy (e.g. 'mean_centered')
- `baseline_channel` int - Channel index used as the baseline when computing relative features

In [ ]:
# --- Parameters ---
_reshape_ml__data_var = 'Sv'
_reshape_ml__dataset_name = 'ml_dataset'
_reshape_ml__feature_strategy = 'mean_centered'
_reshape_ml__baseline_channel = 0

with tracker.step('reshape_ml', op='reshape_for_ml', params={
    'data_var': _reshape_ml__data_var,
    'dataset_name': _reshape_ml__dataset_name,
    'feature_strategy': _reshape_ml__feature_strategy,
    'baseline_channel': _reshape_ml__baseline_channel,
}):
    reshape_ml_ds_ml_ready = reshape_data_for_ml(
    ds_Sv=ds_MVBS,
    data_var=_reshape_ml__data_var,
    dataset_name=_reshape_ml__dataset_name,
    feature_strategy=_reshape_ml__feature_strategy,
    baseline_channel=_reshape_ml__baseline_channel,
)

### Step: `add_aux_features`
**Op:** `add_auxiliary_features`

Append auxiliary coordinate-derived features (e.g. depth, ping_time_seconds) to the flattened ML-ready Dataset. Optional step; omit when no auxiliary features are needed.

**Parameters:**
- `dataset_name` str
- `features` list - List of built-in feature name strings to append. Supported built-ins: 'depth', 'ping_time_seconds', 'seafloor_depth', 'altitude', 'cell_cv'. 'seafloor_depth' and 'altitude' require the echodata input. 'cell_cv' requires the ds_sv input (from compute_per_cell_statistics).


In [ ]:
# --- Parameters ---
_add_aux_features__dataset_name = 'ml_dataset'
_add_aux_features__features = ['cell_cv']

with tracker.step('add_aux_features', op='add_auxiliary_features', params={
    'dataset_name': _add_aux_features__dataset_name,
    'features': _add_aux_features__features,
}):
    add_aux_features_ds_ml_ready = add_auxiliary_features(
    ds_ml_ready=reshape_ml_ds_ml_ready,
    echodata=echodata,
    ds_sv=compute_cell_stats_ds_Sv,
    dataset_name=_add_aux_features__dataset_name,
    features=_add_aux_features__features,
)

### Step: `normalize_ml`
**Op:** `normalize_ml_data`

Normalize ML features in the flattened Dataset. Supports per-feature normalization methods via per_group_methods.

**Parameters:**
- `method` str - Global normalization method (e.g. 'flatten', 'minmax', 'zscore')
- `dataset_name` str
- `normalization_name` str - Label assigned to this normalization for downstream tracking
- `shift_positive` bool - Shift all values to be non-negative after normalization
- `feature_weights` list - Per-feature weight multipliers applied before normalization
- `per_group_methods` dict - Map of feature name to individual normalization method override

In [ ]:
# --- Parameters ---
_normalize_ml__method = 'flatten'
_normalize_ml__dataset_name = 'ml_dataset'
_normalize_ml__normalization_name = 'normalized_flatten_mean_centered'
_normalize_ml__shift_positive = False

with tracker.step('normalize_ml', op='normalize_ml_data', params={
    'method': _normalize_ml__method,
    'dataset_name': _normalize_ml__dataset_name,
    'normalization_name': _normalize_ml__normalization_name,
    'shift_positive': _normalize_ml__shift_positive,
}):
    normalize_ml_ds_normalized = normalize_data(
    ds_ml_ready=add_aux_features_ds_ml_ready,
    method=_normalize_ml__method,
    dataset_name=_normalize_ml__dataset_name,
    normalization_name=_normalize_ml__normalization_name,
    shift_positive=_normalize_ml__shift_positive,
)

### Step: `plot_normalized_ml`
**Op:** `plot_ml_echogram`

Render a normalized ML feature echogram for inspection. This is a sink step: it produces a side-effect (figure) with no data output.

**Parameters:**
- `dataset_name` str
- `normalization_name` str
- `min_depth` float
- `max_depth` float
- `ping_min` int
- `ping_max` int
- `x_axis_units` str
- `y_axis_units` str
- `save_image` bool - Save the rendered figure to an image file; null uses runtime defaults
- `save_formats` list - Image formats to save, chosen from png, svg, jpeg, jpg, and pdf
- `save_dir` str - Optional directory override for saved image artifacts
- `show` bool - Whether to display the figure interactively or inline; null uses runtime defaults
- `dpi` int - Optional DPI override for raster image outputs

In [ ]:
# --- Parameters ---
_plot_normalized_ml__dataset_name = 'ml_dataset'
_plot_normalized_ml__normalization_name = 'normalized_flatten_mean_centered'
_plot_normalized_ml__min_depth = 0
_plot_normalized_ml__max_depth = 1800
_plot_normalized_ml__ping_min = 0
_plot_normalized_ml__ping_max = 515
_plot_normalized_ml__x_axis_units = 'seconds'
_plot_normalized_ml__y_axis_units = 'meters'

with tracker.step('plot_normalized_ml', op='plot_ml_echogram', params={
    'dataset_name': _plot_normalized_ml__dataset_name,
    'normalization_name': _plot_normalized_ml__normalization_name,
    'min_depth': _plot_normalized_ml__min_depth,
    'max_depth': _plot_normalized_ml__max_depth,
    'ping_min': _plot_normalized_ml__ping_min,
    'ping_max': _plot_normalized_ml__ping_max,
    'x_axis_units': _plot_normalized_ml__x_axis_units,
    'y_axis_units': _plot_normalized_ml__y_axis_units,
}):
    plot_flattened_data_echogram(
    ds_ml=normalize_ml_ds_normalized,
    ds_Sv_original=remove_noise_ds_Sv,
    ml_dataset_name=_plot_normalized_ml__dataset_name,
    ml_specific_data_name=_plot_normalized_ml__normalization_name,
    min_depth=_plot_normalized_ml__min_depth,
    max_depth=_plot_normalized_ml__max_depth,
    ping_min=_plot_normalized_ml__ping_min,
    ping_max=_plot_normalized_ml__ping_max,
    x_axis_units=_plot_normalized_ml__x_axis_units,
    y_axis_units=_plot_normalized_ml__y_axis_units,
)

### Step: `run_hdbscan`
**Op:** `run_hdbscan`

Run HDBSCAN unsupervised clustering on normalized feature data. Pure-compute step: accepts normalized features and returns a clustering result dict without modifying any Dataset. Designed to be a sweep target for hyperparameter comparison.

**Parameters:**
- `dataset_name` str
- `normalization_name` str
- `ml_result_name` str - Label assigned to this clustering result for downstream tracking
- `min_cluster_size` int - Absolute minimum cluster size. Mutually exclusive with min_cluster_size_fraction.
- `min_cluster_size_fraction` float - Fraction of extracted data points used to derive min_cluster_size when min_cluster_size is omitted. Mutually exclusive with min_cluster_size. When omitted, the ML function defaults to 0.03.

- `min_samples` int
- `sample_size` int - Maximum number of samples drawn for clustering;
- `cluster_selection_method` str - HDBSCAN cluster selection method ('eom' or 'leaf')
- `use_hdbscan` bool - Use HDBSCAN; set false to fall back to a simpler clustering method
- `epsilon` float - DBSCAN epsilon distance parameter
- `find_background_cluster` bool - Run background-cluster detection path instead of standard HDBSCAN. When true, background_label output is populated.

- `soft_membership_threshold` float - Reassign noise points via soft-membership probabilities when set. null disables soft-membership reassignment.


In [ ]:
# --- Parameters ---
_run_hdbscan__dataset_name = 'ml_dataset'
_run_hdbscan__normalization_name = 'normalized_flatten_mean_centered'
_run_hdbscan__ml_result_name = 'hdbscan_results'
_run_hdbscan__min_cluster_size_fraction = 0.02
_run_hdbscan__min_samples = 10
_run_hdbscan__sample_size = 1000000
_run_hdbscan__cluster_selection_method = 'leaf'
_run_hdbscan__use_hdbscan = True

with tracker.step('run_hdbscan', op='run_hdbscan', params={
    'dataset_name': _run_hdbscan__dataset_name,
    'normalization_name': _run_hdbscan__normalization_name,
    'ml_result_name': _run_hdbscan__ml_result_name,
    'min_cluster_size_fraction': _run_hdbscan__min_cluster_size_fraction,
    'min_samples': _run_hdbscan__min_samples,
    'sample_size': _run_hdbscan__sample_size,
    'cluster_selection_method': _run_hdbscan__cluster_selection_method,
    'use_hdbscan': _run_hdbscan__use_hdbscan,
}):
    _result = run_hdbscan(
    ds_normalized=normalize_ml_ds_normalized,
    dataset_name=_run_hdbscan__dataset_name,
    normalization_name=_run_hdbscan__normalization_name,
    ml_result_name=_run_hdbscan__ml_result_name,
    min_cluster_size_fraction=_run_hdbscan__min_cluster_size_fraction,
    min_samples=_run_hdbscan__min_samples,
    sample_size=_run_hdbscan__sample_size,
    cluster_selection_method=_run_hdbscan__cluster_selection_method,
    use_hdbscan=_run_hdbscan__use_hdbscan,
)
    clustering_results = _result['clustering_results']
    clustering_model = _result['clustering_model']
    background_label = _result['background_label']
    cluster_labels = _result['cluster_labels']

### Step: `embed_results`
**Op:** `embed_clustering_results`

Embed HDBSCAN clustering results into the normalized ML Dataset. Accepts a single result dict from run_hdbscan, or a collected list of result dicts when used with the collect pattern for ensemble runs. For a list, one cluster-label variable is embedded per collected result using deterministic names derived from ml_result_name.

**Parameters:**
- `dataset_name` str
- `ml_result_name` str

In [ ]:
# --- Parameters ---
_embed_results__dataset_name = 'ml_dataset'
_embed_results__ml_result_name = 'hdbscan_results'

with tracker.step('embed_results', op='embed_clustering_results', params={
    'dataset_name': _embed_results__dataset_name,
    'ml_result_name': _embed_results__ml_result_name,
}):
    _result = embed_clustering_results(
    ds_normalized=normalize_ml_ds_normalized,
    clustering_results=clustering_results,
    dataset_name=_embed_results__dataset_name,
    ml_result_name=_embed_results__ml_result_name,
)
    embed_results_ds_normalized = _result['ds_normalized']
    gridded_results = _result['gridded_results']

### Step: `plot_clustering_report`
**Op:** `plot_clustering_report`

Render a full clustering report for one or more clustering results. Produces the clustering echogram, per-cluster statistics report, and the HDBSCAN hierarchy plot when the clustering result includes a model and hierarchy plotting is applicable. This is a sink step: it produces figures with no data output.

**Parameters:**
- `dataset_name` str
- `ml_result_name` str
- `plot_window` list - [min_depth, max_depth, ping_min, ping_max] for echogram display
- `overlay_line_var` str - Name of a variable in ds_normalized to draw as a line overlay
- `cluster_colors` list - Hex color strings for each cluster label
- `y_to_x_aspect_ratio_override` float - Override the echogram aspect ratio; null uses the default
- `cluster_stats_sv_data_var` str - Sv variable name used when computing per-cluster statistics, or "ml_features" to plot the flattened ML features used for clustering
- `cluster_stats_compute_pairwise_differences` bool - Compute inter-channel pairwise diffs in cluster statistics
- `save_image` bool - Save the rendered figures to image files; null uses runtime defaults
- `save_formats` list - Image formats to save, chosen from png, svg, jpeg, jpg, and pdf
- `save_dir` str - Optional directory override for saved image artifacts
- `show` bool - Whether to display figures interactively or inline; null uses runtime defaults
- `dpi` int - Optional DPI override for raster image outputs

In [ ]:
# --- Parameters ---
_plot_clustering_report__dataset_name = 'ml_dataset'
_plot_clustering_report__ml_result_name = 'hdbscan_results'
_plot_clustering_report__plot_window = [0, 1800, 0, 515]
_plot_clustering_report__overlay_line_var = 'sw_dive_profile'
_plot_clustering_report__cluster_colors = ['#9D00FF', '#35E200', '#FF0000', '#2F00FF', '#FF8800', '#FFFB1C', '#FF44E9', '#00BE8BCA', '#00FFE5', '#3D8D00FF', '#9C67FFFF']
_plot_clustering_report__cluster_stats_sv_data_var = 'Sv'

with tracker.step('plot_clustering_report', op='plot_clustering_report', params={
    'dataset_name': _plot_clustering_report__dataset_name,
    'ml_result_name': _plot_clustering_report__ml_result_name,
    'plot_window': _plot_clustering_report__plot_window,
    'overlay_line_var': _plot_clustering_report__overlay_line_var,
    'cluster_colors': _plot_clustering_report__cluster_colors,
    'cluster_stats_sv_data_var': _plot_clustering_report__cluster_stats_sv_data_var,
}):
    plot_clustering_report(
    ds_normalized=embed_results_ds_normalized,
    clustering_results=clustering_results,
    clustering_model=clustering_model,
    ds_Sv=remove_noise_ds_Sv,
    dataset_name=_plot_clustering_report__dataset_name,
    ml_result_name=_plot_clustering_report__ml_result_name,
    plot_window=_plot_clustering_report__plot_window,
    overlay_line_var=_plot_clustering_report__overlay_line_var,
    cluster_colors=_plot_clustering_report__cluster_colors,
    cluster_stats_sv_data_var=_plot_clustering_report__cluster_stats_sv_data_var,
)

*End of included section: processing_lvl_4.yaml*

In [ ]:
tracker.save_recipe('pipeline_modified.yaml')

In [ ]:
# Capture the runtime environment and save provenance to YAML.
import io as _prov_io
from pathlib import Path as _ProvPath
from ruamel.yaml import YAML as _ProvYAML
_provenance = ProvenanceRecorder.capture_environment(['aa-si-utils', 'echopype', 'aa-si-ml', 'aa-si-visualization'])
_provenance_dir = _ProvPath('outputs/provenance')
_provenance_dir.mkdir(parents=True, exist_ok=True)
_prov_yaml = _ProvYAML()
_prov_yaml.default_flow_style = False
_prov_stream = _prov_io.StringIO()
_prov_yaml.dump(_provenance, _prov_stream)
(_provenance_dir / 'provenance.yaml').write_text(_prov_stream.getvalue(), encoding='utf-8')
print("Python:", _provenance["python_version_number"])
print("Timestamp:", _provenance["timestamp"])
if "installed_packages" in _provenance:
    for pkg, ver in _provenance["installed_packages"].items():
        print(f"  {pkg}: {ver}")
print(f"Provenance saved to: {_provenance_dir / 'provenance.yaml'}")